# Notebook 15 — Agents & Workflows

Build an orchestrated multi-step agent that decides which tools to use and when to stop.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from utils.anthropic_client import client, FAST_MODEL


## Workflow vs Agent

| | Workflow | Agent |
|---|---|---|
| Control flow | Hard-coded | LLM decides |
| Flexibility | Low | High |
| Reliability | High | Medium |
| Use case | Known steps | Open-ended tasks |

## A simple research agent

In [ ]:
import math, re, json

# Tools
def search(query):
    MOCK_DB = {
        "python": "Python is a high-level language created by Guido van Rossum in 1991.",
        "javascript": "JavaScript is a scripting language created by Brendan Eich in 1995.",
        "rust": "Rust is a systems language by Mozilla focused on memory safety, released in 2010.",
    }
    for key, val in MOCK_DB.items():
        if key in query.lower():
            return val
    return "No results found."

def summarize(text):
    r = client.messages.create(model=FAST_MODEL, max_tokens=80,
        messages=[{"role":"user","content":f"Summarize in one sentence: {text}"}])
    return r.content[0].text

TOOL_LIST = [
    {"name":"search",    "description":"Search for info on a programming language.",
     "input_schema":{"type":"object","properties":{"query":{"type":"string"}},"required":["query"]}},
    {"name":"summarize", "description":"Summarize a passage of text into one sentence.",
     "input_schema":{"type":"object","properties":{"text":{"type":"string"}},"required":["text"]}},
]

def dispatch(name, inputs):
    if name == "search":    return search(inputs["query"])
    if name == "summarize": return summarize(inputs["text"])
    return "unknown tool"


## Run the agent

In [ ]:
def run_research_agent(task, max_steps=6):
    messages = [{"role":"user","content":task}]
    print(f"Task: {task}\n")
    for step in range(max_steps):
        r = client.messages.create(model=FAST_MODEL, max_tokens=512, tools=TOOL_LIST, messages=messages)
        messages.append({"role":"assistant","content":r.content})
        if r.stop_reason == "end_turn":
            answer = next((b.text for b in r.content if hasattr(b,"text")), "")
            print(f"Answer: {answer}")
            return answer
        if r.stop_reason == "tool_use":
            results = []
            for block in r.content:
                if block.type == "tool_use":
                    out = dispatch(block.name, block.input)
                    print(f"  [{block.name}] {str(block.input)[:60]} -> {str(out)[:60]}")
                    results.append({"type":"tool_result","tool_use_id":block.id,"content":out})
            messages.append({"role":"user","content":results})
    return "max steps reached"

run_research_agent("Search for Python and JavaScript, summarize each, then compare them in one paragraph.")


## Exercise
Add a third tool (e.g., `compare_languages`) and test the agent on a 3-step task.

In [ ]:
# Your code here
